In [1]:
!pip install imblearn


   ------------- -------------------------- 1/3 [imbalanced-learn]
   ------------- -------------------------- 1/3 [imbalanced-learn]
   ---------------------------------------- 3/3 [imblearn]




[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
from collections import Counter

# Pré-processamento e avaliação
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

# Modelo
from sklearn.tree import DecisionTreeClassifier

# SMOTE e Pipeline do imblearn
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# 1) Ler a base (ajuste o caminho se necessário)
#path = "/mnt/data/base_exemplo_classificacao.csv"
df = pd.read_csv("Balanceamento_data_base.csv")

# 2) Definir X (features) e y (target)
#    Alvo binário 'Prefere_TP' (por exemplo: 'Sim'/'Nao')
target_col = "Prefere_TP"
X = df.drop(columns=[target_col])
y = df[target_col]

# 3) Separar treino e teste (estratificando para preservar proporções)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# 4) Identificar colunas numéricas e categóricas
num_cols = X_train.select_dtypes(include="number").columns.tolist()
cat_cols = X_train.select_dtypes(exclude="number").columns.tolist()

# 5) Transformador de colunas:
#    - Numéricas: StandardScaler (não é essencial para árvore, mas não atrapalha)
#    - Numéricas: alternativa é "passthrough" (sem escala)
#    - Categóricas: OneHotEncoder (ignora categorias desconhecidas)
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(with_mean=False), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols), #handle_unknown="ignore": categorias novas → linha de zeros.
    ]
)

# 6) Definir k_neighbors do SMOTE de forma robusta (evita erro com poucas amostras na minoria)
min_count = y_train.value_counts().min()
k_neighbors = max(1, min(5, min_count - 1))  # k <= (amostras_minoritárias - 1) e no máx. 5

# 7) Montar o pipeline: preprocessamento -> SMOTE -> Árvore de Decisão
pipe = ImbPipeline(
    steps=[
        ("preprocess", preprocess), # Aplica ColumnTransformer 
        ("smote", SMOTE(random_state=42, k_neighbors=k_neighbors)), # Oversampling no treino
        ("model", DecisionTreeClassifier( # Modelo: Árvore de Decisão
            criterion="gini",      # ou "entropy"
            max_depth=None,        # Profundidade ilimitada (pode ajustar para evitar overfitting)
            min_samples_split=2,   # Mínimo de amostras para dividir um nó
            min_samples_leaf=1,    # Mínimo de amostras em uma folha
            random_state=42        # Reprodutibilidade da árvore
        )),
    ]
)

# 8) Treinar o pipeline (SMOTE é aplicado SOMENTE no treino dentro do pipeline)
pipe.fit(X_train, y_train)

# 9) Avaliação no conjunto de teste
y_pred = pipe.predict(X_test)
print("Relatório de classificação (teste):\n")
print(classification_report(y_test, y_pred))
print("Matriz de confusão (linhas = verdade, colunas = predição):")
print(confusion_matrix(y_test, y_pred), "\n")

# 10) Inspecionar o balanceamento NO TREINO após o SMOTE
preprocess_fitted = pipe.named_steps["preprocess"]
X_train_pp = preprocess_fitted.transform(X_train)
y_train_counts = Counter(y_train)

smote_for_inspect = SMOTE(random_state=42, k_neighbors=k_neighbors)
X_res, y_res = smote_for_inspect.fit_resample(X_train_pp, y_train)
y_res_counts = Counter(y_res)

print("Distribuição de classes no treino (antes SMOTE) :", dict(y_train_counts))
print("Distribuição de classes no treino (após  SMOTE) :", dict(y_res_counts))

# Observações:
# - O Pipeline do imblearn garante que o SMOTE NÃO veja o conjunto de teste (evita data leakage).
# - Você pode ajustar hiperparâmetros da árvore (max_depth, min_samples_leaf, etc.) ou
#   acoplar um GridSearchCV sobre 'pipe' para otimização, sem mudar o fluxo.

Relatório de classificação (teste):

              precision    recall  f1-score   support

         Nao       1.00      0.96      0.98        28
         Sim       0.67      1.00      0.80         2

    accuracy                           0.97        30
   macro avg       0.83      0.98      0.89        30
weighted avg       0.98      0.97      0.97        30

Matriz de confusão (linhas = verdade, colunas = predição):
[[27  1]
 [ 0  2]] 

Distribuição de classes no treino (antes SMOTE) : {'Nao': 65, 'Sim': 5}
Distribuição de classes no treino (após  SMOTE) : {'Nao': 65, 'Sim': 65}
